# 02 — Feature Extraction

This notebook covers the multimodal feature extraction pipelines (Chapter 4).

**Pipeline overview:**
1. **Video Representations** — VideoMAE-v2 ViT-Giant embeddings (D=1408) from 16-frame windows
2. **Speech** — WhisperX ASR transcription + multilingual-e5-large sentence embeddings (D=768)
3. **Hand Landmarks** — MediaPipe hand landmark detection (21 landmarks/hand) + temporal tracking
4. **Skeleton Pose** — MediaPipe pose landmark detection (33 landmarks) at 0.5 Hz sampling
5. **Gaze** — Tobii eye-tracking data parsing (placeholder — extraction not yet implemented)
6. **Running on HPC** — SLURM batch scripts for cluster execution

In [ ]:
import json
import os
from glob import glob

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import display

from smartflat.datasets.dataset_gaze import parse_tobii_data
from smartflat.datasets.loader import get_dataset
from smartflat.features.audio.main import main as main_audio
from smartflat.features.audio.main import parse_json as parse_audio_json
from smartflat.features.hands.main import create_hand_video, main as main_hands
from smartflat.features.hands.main import parse_json as parse_hand_json
from smartflat.features.hands_processing.main import main as main_hands_processing
from smartflat.features.skeleton.main import create_skeleton_video, main as main_skeleton
from smartflat.features.video.main import main as main_video
from smartflat.utils.utils_io import fetch_output_path, get_data_root

In [ ]:
# Data root — resolved via hostname or SMARTFLAT_DATA_ROOT env var
data_root = get_data_root()
print(f"Data root: {data_root}")

## 1. Video Representations (VideoMAE-v2)

Extract ViT-Giant-Patch14 embeddings (D=1408) from sliding 16-frame windows with stride 8.
Each video yields an `[N, 1408]` numpy array where N depends on video length. The model
requires a GPU (A100) and the VideoMAE-v2 checkpoint.

**Key parameters:** `segment_length=16`, `delta_t=8`, `model=vit_giant_patch14_224`

See: `smartflat.features.video.main`

In [ ]:
# Dataset: videos in the gold set that have not yet been processed
dset_video = get_dataset(dataset_name='video_block_representation', scenario='gold_unprocessed')
df_video = dset_video.metadata.copy()
print(f"Videos awaiting extraction: {len(df_video)}")

# Uncomment to run (requires A100 GPU with CUDA, ~24h for full dataset):
# main_video(device='0', chunk_idx=0, num_chunks=10)

In [ ]:
# Verify a sample processed embedding
dset_video_all = get_dataset(dataset_name='video_block_representation', scenario='gold')
df_video_all = dset_video_all.metadata.copy()

sample_found = False
for _, row in df_video_all.iterrows():
    output_path = fetch_output_path(row['video_path'], 'vit_giant_patch14_224')
    if os.path.isfile(output_path):
        embedding = np.load(output_path)
        print(f"Sample: {row['identifier']}")
        print(f"  Embedding path: {output_path}")
        print(f"  Shape: {embedding.shape}  (expected: [N, 1408])")
        print(f"  dtype: {embedding.dtype}")
        print(f"  Value range: [{embedding.min():.3f}, {embedding.max():.3f}]")
        print(f"  Mean: {embedding.mean():.3f}, Std: {embedding.std():.3f}")
        sample_found = True
        break

if not sample_found:
    print("No processed video embeddings found. Run main_video() first.")

In [ ]:
# Gram matrix (self-similarity) of a sample video embedding
if sample_found:
    gram = embedding @ embedding.T
    fig, ax = plt.subplots(figsize=(8, 6))
    im = ax.imshow(gram, aspect='auto', cmap='viridis')
    ax.set(xlabel='Segment index', ylabel='Segment index',
           title=f'Video embedding Gram matrix — {row["identifier"]}\n'
                 f'Shape: {embedding.shape[0]} segments × D={embedding.shape[1]}')
    plt.colorbar(im, ax=ax, label='Cosine similarity (unnormalized)')
    plt.tight_layout()
    plt.show()
else:
    print("Skipping Gram matrix — no embedding available.")

## 2. Speech (WhisperX + e5-large)

Two-stage pipeline: (1) WhisperX ASR produces JSON transcripts with word-level timestamps,
confidence scores, and speaker diarization; (2) multilingual-e5-large encodes each utterance
into a 768-dimensional sentence embedding. Output: `[N, 768]` numpy array per recording.

See: `smartflat.features.audio.main`

In [ ]:
# Dataset: recordings awaiting speech extraction
dset_speech = get_dataset(dataset_name='speech_recognition_representation', scenario='gold_unprocessed')
df_speech = dset_speech.metadata.copy()
print(f"Recordings awaiting speech extraction: {len(df_speech)}")

# Uncomment to run (requires WhisperX binary + GPU):
# main_audio(whisperx_bin_file='/path/to/whisperx')

In [ ]:
# Verify sample speech outputs: WhisperX JSON transcript + e5-large embeddings
dset_speech_all = get_dataset(dataset_name='speech_recognition_representation', scenario='gold')
df_speech_all = dset_speech_all.metadata.copy()

speech_sample_found = False
for _, row in df_speech_all.iterrows():
    json_path = fetch_output_path(row['video_path'], 'whisperx')
    emb_path = fetch_output_path(row['video_path'], 'multilingual-e5-large')

    if os.path.isfile(json_path) and os.path.isfile(emb_path):
        # Parse transcript
        starts, ends, texts, confidences, speakers = parse_audio_json(json_path)
        print(f"Sample: {row['identifier']}")
        print(f"\n--- WhisperX transcript ({len(texts)} segments) ---")
        for i in range(min(5, len(texts))):
            conf = np.mean(confidences[i]) if confidences[i] else 0.0
            print(f"  [{starts[i]:.1f}s – {ends[i]:.1f}s] (conf={conf:.2f}) {texts[i]}")
        if len(texts) > 5:
            print(f"  ... ({len(texts) - 5} more segments)")

        # Load embeddings
        speech_emb = np.load(emb_path)
        print(f"\n--- Speech embeddings ---")
        print(f"  Shape: {speech_emb.shape}  (expected: [N, 768])")
        print(f"  dtype: {speech_emb.dtype}")
        print(f"  Value range: [{speech_emb.min():.3f}, {speech_emb.max():.3f}]")
        speech_sample_found = True
        break

if not speech_sample_found:
    print("No processed speech data found. Run main_audio() first.")

## 3. Hand Landmarks (MediaPipe)

Two-step pipeline: (1) MediaPipe Hand Landmarker detects up to 4 hands per frame, producing
21 landmarks each with 3D coordinates, handedness, and confidence scores; (2) post-processing
applies confidence filtering (threshold=0.9), joins nearby detections (gap ≤ 0.5s), and
discards short segments (< 0.5s) to build temporal hand tracking segments.

See: `smartflat.features.hands.main`, `smartflat.features.hands_processing.main`

In [ ]:
# Dataset: videos awaiting hand landmark extraction
dset_hands = get_dataset(dataset_name='hand_landmarks', scenario='gold_unprocessed')
df_hands = dset_hands.metadata.copy()
print(f"Videos awaiting hand landmark extraction: {len(df_hands)}")

# Uncomment to run (CPU-only, ~24h for full dataset):
# Step 1: Hand landmark detection (MediaPipe)
# main_hands(root_dir=None)

# Step 2: Temporal tracking post-processing (confidence filtering + segment joining)
# main_hands_processing(root_dir=None)

In [ ]:
# Verify sample hand landmarks: parse JSON, show left/right detection rates
dset_hands_all = get_dataset(dataset_name='hand_landmarks', scenario='gold')
df_hands_all = dset_hands_all.metadata.copy()

hands_sample_found = False
for _, row in df_hands_all.iterrows():
    hand_path = fetch_output_path(row['video_path'], 'hand_landmarks_mediapipe')
    if os.path.isfile(hand_path):
        # Parse into left/right DataFrames
        df_left, df_right = parse_hand_json(hand_path)
        n_frames = max(len(df_left), len(df_right))
        left_detected = df_left['detected'].sum() if 'detected' in df_left.columns else 0
        right_detected = df_right['detected'].sum() if 'detected' in df_right.columns else 0

        print(f"Sample: {row['identifier']}")
        print(f"  Landmarks path: {hand_path}")
        print(f"  Total frames: {n_frames}")
        print(f"  Left hand detected:  {left_detected}/{n_frames} "
              f"({left_detected/n_frames:.1%})" if n_frames > 0 else "")
        print(f"  Right hand detected: {right_detected}/{n_frames} "
              f"({right_detected/n_frames:.1%})" if n_frames > 0 else "")
        hands_sample_found = True
        break

if not hands_sample_found:
    print("No processed hand landmarks found. Run main_hands() first.")

In [ ]:
# Uncomment to create an annotated hand video (requires video file + decord):
# if hands_sample_found:
#     frames = create_hand_video(
#         identifier=row['identifier'],
#         hand_landmarks_path=hand_path,
#         video_path=row['video_path'],
#         downsampling_factor=25,
#         overwrite=False,
#     )

## 4. Skeleton Pose (MediaPipe)

Extract 33 pose landmarks per frame using MediaPipe PoseLandmarker (heavy model). Frames are
sampled at `sampling_frequency=0.5` Hz (one detection every 0.5 seconds) to reduce compute.
Up to 3 poses are detected per frame. Output: JSON with pose landmarks and world landmarks.

See: `smartflat.features.skeleton.main`

In [ ]:
# Dataset: videos awaiting skeleton extraction
dset_skeleton = get_dataset(dataset_name='skeleton_landmarks', scenario='gold_unprocessed')
df_skeleton = dset_skeleton.metadata.copy()
print(f"Videos awaiting skeleton extraction: {len(df_skeleton)}")

# Uncomment to run (CPU-only, ~72h for full dataset):
# main_skeleton()

In [ ]:
# Verify sample skeleton landmarks
dset_skeleton_all = get_dataset(dataset_name='skeleton_landmarks', scenario='gold')
df_skeleton_all = dset_skeleton_all.metadata.copy()

skeleton_sample_found = False
for _, row in df_skeleton_all.iterrows():
    skel_path = fetch_output_path(row['video_path'], 'skeleton_landmarks_mediapipe')
    if os.path.isfile(skel_path):
        with open(skel_path, 'r') as f:
            skel_data = json.load(f)

        n_sampled_frames = len(skel_data)
        n_with_pose = sum(1 for frame in skel_data if frame.get('pose_landmarks'))

        print(f"Sample: {row['identifier']}")
        print(f"  Landmarks path: {skel_path}")
        print(f"  Sampled frames: {n_sampled_frames}")
        print(f"  Frames with pose detected: {n_with_pose}/{n_sampled_frames} "
              f"({n_with_pose/n_sampled_frames:.1%})" if n_sampled_frames > 0 else "")

        # Show structure of first detection
        if n_with_pose > 0:
            first_with_pose = next(f for f in skel_data if f.get('pose_landmarks'))
            n_poses = len(first_with_pose['pose_landmarks'])
            n_landmarks = len(first_with_pose['pose_landmarks'][0]) if n_poses > 0 else 0
            print(f"  Poses per frame: {n_poses}  (max_num_poses=3)")
            print(f"  Landmarks per pose: {n_landmarks}  (expected: 33)")

        skeleton_sample_found = True
        break

if not skeleton_sample_found:
    print("No processed skeleton landmarks found. Run main_skeleton() first.")

In [ ]:
# Uncomment to create an annotated skeleton video (requires video file + decord):
# if skeleton_sample_found:
#     frames = create_skeleton_video(
#         identifier=row['identifier'],
#         skeleton_landmarks_path=skel_path,
#         video_path=row['video_path'],
#         overwrite=False,
#     )

## 5. Gaze (Tobii) — Placeholder

**Status:** `smartflat.features.gaze.main` raises `NotImplementedError` — it was templated from
the video module but never completed. This is a known gap.

The raw Tobii eye-tracking data **can** be parsed via `smartflat.datasets.dataset_gaze.parse_tobii_data()`,
which returns four DataFrames: gaze coordinates (2D/3D), gaze events (fixations/saccades),
accelerometer, and gyroscope data. The integration into a feature extraction pipeline (comparable
to video/speech/hands) remains future work.

See: `smartflat.datasets.dataset_gaze.parse_tobii_data`

In [ ]:
# NOTE: smartflat.features.gaze.main raises NotImplementedError at import time.
# The gaze feature extraction module was templated from the video module but never
# completed. Tobii raw data parsing is available via dataset_gaze.parse_tobii_data().
#
# Future work: implement a gaze feature extraction pipeline that converts raw Tobii
# TSV files into per-segment gaze features (fixation counts, saccade rates, etc.).

In [ ]:
# Demo: parse a Tobii recording to show available data types
# Find a Tobii recording in the base dataset
dset_base = get_dataset(dataset_name='base', scenario='all')
df_base = dset_base.metadata.copy()
df_tobii = df_base[df_base['modality'] == 'Tobii']

gaze_sample_found = False
if len(df_tobii) > 0:
    for _, row in df_tobii.iterrows():
        # Tobii TSV files are typically in the recording directory
        tobii_dir = os.path.dirname(row['video_path']) if pd.notna(row.get('video_path')) else None
        if tobii_dir and os.path.isdir(tobii_dir):
            tsv_files = glob(os.path.join(tobii_dir, '*.tsv'))
            if tsv_files:
                tobii_file = tsv_files[0]
                print(f"Sample: {row['identifier']}")
                print(f"  Tobii file: {tobii_file}\n")

                gaze_dict = parse_tobii_data(tobii_file, data_type='all')

                for data_type, df_gaze in gaze_dict.items():
                    if df_gaze is not None and len(df_gaze) > 0:
                        print(f"--- {data_type} ---")
                        print(f"  Shape: {df_gaze.shape}")
                        print(f"  Columns: {list(df_gaze.columns)}")
                        display(df_gaze.head(3))
                        print()
                    else:
                        print(f"--- {data_type} --- (empty)")

                gaze_sample_found = True
                break

if not gaze_sample_found:
    print("No Tobii recordings found in the dataset.")

## 6. Running on HPC

SLURM batch scripts for each modality are in `scripts/compute/`. Each script configures the
appropriate partition (GPU vs CPU), conda environment, and module dependencies. Submit scripts
call the corresponding `python -m smartflat.features.<modality>.main` entry point.

See: `scripts/compute/` for per-modality job scripts

In [ ]:
# HPC configuration summary per modality
hpc_config = pd.DataFrame([
    {'Modality': 'Video (VideoMAE-v2)', 'SLURM Script': 'ruche_gpu.sh',
     'Partition': 'gpua100', 'GPU': '1× A100', 'Conda Env': 'smartflat',
     'Time Limit': '24h', 'Tasks': 4},
    {'Modality': 'Speech (WhisperX)', 'SLURM Script': 'ruche_speech.sh',
     'Partition': 'gpua100', 'GPU': '4× A100', 'Conda Env': 'whisperx',
     'Time Limit': '24h', 'Tasks': 32},
    {'Modality': 'Hands (MediaPipe)', 'SLURM Script': 'ruche_hands.sh',
     'Partition': 'cpu_long', 'GPU': 'None', 'Conda Env': 'mediapipe',
     'Time Limit': '24h', 'Tasks': 4},
    {'Modality': 'Skeleton (MediaPipe)', 'SLURM Script': 'ruche_skeleton.sh',
     'Partition': 'mem', 'GPU': 'None', 'Conda Env': 'mediapipe',
     'Time Limit': '72h', 'Tasks': 80},
])
display(hpc_config.set_index('Modality'))

In [ ]:
# List available HPC scripts
scripts_dir = os.path.join(os.path.dirname(os.path.abspath('.')), 'scripts', 'compute')
if os.path.isdir(scripts_dir):
    scripts = sorted([f for f in os.listdir(scripts_dir) if f.endswith('.sh')])
    print(f"HPC scripts in scripts/compute/ ({len(scripts)} files):")
    for s in scripts:
        print(f"  {s}")
else:
    print(f"Scripts directory not found at {scripts_dir}")

# NOTE: HPC scripts currently use the old import path structure:
#   Old: $SMARTFLAT_ROOT/api/features/<modality>/main.py
#   New: python -m smartflat.features.<modality>.main
# These scripts need updating before the next HPC batch run.
print("\nNOTE: HPC scripts use the old 'api/' import paths.")
print("Update to 'python -m smartflat.features.<modality>.main' before next batch run.")